# Pitch type prediction – XGBoost prototype

Train an XGBoost classifier to predict **pitch_type** (FF, SL, CH, etc.) from count, leverage, previous pitch in AB, and other pitch/context features.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import xgboost as xgb

# Project root on path for src imports
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import statsapi
from src.features import build_pitch_features, add_pitcher_tendency

## Load data

Fetch a game from the MLB API and build pitch features. Optionally load from a CSV if you've run `dataset_generator`.

In [ ]:
def load_training_data(use_csv: bool = True):
    """Load pitch data: from data/*.csv if present and use_csv else fetch one game."""
    data_dir = ROOT / "data"
    if use_csv and data_dir.exists():
        csvs = list(data_dir.glob("pitch_features_*.csv"))
        if csvs:
            path = max(csvs, key=lambda p: p.stat().st_mtime)
            df = pd.read_csv(path)
            print(f"Loaded {len(df)} pitches from {path.name}")
            return df
    # Fallback: fetch one game (Game 7, 2016 World Series)
    game_pk = 487637
    print(f"Fetching game {game_pk} from API...")
    play_data = statsapi.get("game", {"gamePk": game_pk})
    df = build_pitch_features(play_data)
    tendency = add_pitcher_tendency(play_data)
    for key in ["pitcher_tendency_primary_pct", "pitcher_tendency_fastball_pct", "pitcher_tendency_breaking_pct", "pitcher_tendency_offspeed_pct", "pitcher_tendency_pitches_used"]:
        df[key] = [t[key] for t in tendency]
    print(f"Built {len(df)} pitches with features")
    return df

df = load_training_data(use_csv=True)

## Prepare target and features

Drop rows with missing `pitch_type`. Encode `prev_pitch_type_in_ab` (label encoding). Use count one-hot, is_leverage, inning, velocity, spin_rate, and optional pitcher tendency columns.

In [ ]:
df = df.dropna(subset=["pitch_type"]).copy()
df["pitch_type"] = df["pitch_type"].astype(str).str.upper()

# Optional: collapse rare pitch types into "Other" (e.g. keep top 8–10)
MIN_SAMPLES = 20
counts = df["pitch_type"].value_counts()
rare = counts[counts < MIN_SAMPLES].index.tolist()
if rare:
    df.loc[df["pitch_type"].isin(rare), "pitch_type"] = "Other"

y = df["pitch_type"]
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
class_names = label_encoder.classes_.tolist()
print("Target classes:", class_names)
print("Class counts:", pd.Series(y_encoded).value_counts().sort_index().to_dict())

In [ ]:
count_cols = [c for c in df.columns if c.startswith("count_") and "-" not in c]
numeric = ["inning", "balls", "strikes", "outs", "is_leverage", "velocity", "spin_rate", "score_home", "score_away"]
tendency_cols = [c for c in df.columns if c.startswith("pitcher_tendency_") and (c.endswith("_pct") or c == "pitcher_tendency_pitches_used")]

feature_cols = count_cols + [c for c in numeric if c in df.columns] + tendency_cols

# Encode prev_pitch_type_in_ab (fill missing with "None")
prev = df["prev_pitch_type_in_ab"].fillna("None").astype(str).str.upper()
prev_enc = LabelEncoder().fit_transform(prev)
df["prev_pitch_type_in_ab_enc"] = prev_enc
feature_cols = ["prev_pitch_type_in_ab_enc"] + [c for c in feature_cols if c in df.columns]

X = df[feature_cols].fillna(0)
print("Feature matrix shape:", X.shape)
print("Features:", feature_cols)

## Train / test split and XGBoost

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    use_label_encoder=False,
    eval_metric="mlogloss",
)
model.fit(X_train, y_train)

## Evaluate

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")
print()
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
print("Confusion matrix (rows=true, cols=pred):")
try:
    display(cm_df)
except NameError:
    print(cm_df.to_string())

In [ ]:
import joblib

joblib.dump(model, "models/pitch_predictor.pkl")